In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")

In [5]:
def normalize_day_type(
    df,
    day_type_col,
    day_of_week_col
):
    """
    Reclassifies 'Holiday' rows into Weekday/Saturday/Sunday
    based on the day of week.
    """
    return df.apply(
        lambda row: (
            'Weekday'
            if row[day_type_col] == 'Holiday'
            and row[day_of_week_col] in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
            else (
                'Saturday'
                if row[day_type_col] == 'Holiday'
                and row[day_of_week_col] == 'Saturday'
                else (
                    'Sunday'
                    if row[day_type_col] == 'Holiday'
                    and row[day_of_week_col] == 'Sunday'
                    else row[day_type_col]
                )
            )
        ),
        axis=1
    )

In [6]:
# Function to count number of days of a Day_Type in a service period
def count_day_type_days(start, end, day_type):
    dates = pd.date_range(start, end)
    day_type_upper = day_type.upper()
    if day_type_upper == 'WEEKDAY':
        return sum(d.weekday() < 5 for d in dates)  
    elif day_type_upper == 'SATURDAY':
        return sum(d.weekday() == 5 for d in dates)
    elif day_type_upper == 'SUNDAY':
        return sum(d.weekday() == 6 for d in dates)
    else:
        return 0

In [7]:
def add_day_type(df, date_col, output_col='Day Type'):
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [8]:
bart_ridership['Day Type'] = normalize_day_type(
    bart_ridership,
    day_type_col='Day Type',
    day_of_week_col='Day of Week'
)

bart_ridership['exposure'] = bart_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

bart_ridership.rename(columns={
    'Station': 'Stop',
    'Station Name': 'Stop Name',
    'Entries': 'Boardings'
}, inplace=True)

In [9]:
# Compute exposure: number of days of that Day_Type in the period
big_blue_bus_ridership['exposure'] = big_blue_bus_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['SERVICE_DAY']),
    axis=1
)

# Compute total boardings for the period
big_blue_bus_ridership['Boardings'] = big_blue_bus_ridership['AVERAGE_DAILY_BOARDINGS'] * big_blue_bus_ridership['exposure']

# Rename columns
big_blue_bus_ridership.rename(columns={
    'SERVICE_DAY': 'Day_Type',
    'STOP_ID': 'Stop',
    'STOP_NAME': 'Stop Name'
}, inplace=True)


For Caltrain Ridership, dropping "Holiday" data since we just have average ridership for the month. 

In [10]:
caltrain_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2520 entries, 0 to 2519
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Month               2520 non-null   object        
 1   Origin Station      2520 non-null   object        
 2   Caltrain Ridership  2520 non-null   float64       
 3   Date Type           2520 non-null   object        
 4   Average Ridership   2520 non-null   float64       
 5   start_date          2520 non-null   datetime64[ns]
 6   end_date            2520 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), object(3)
memory usage: 137.9+ KB


In [11]:
caltrain_ridership = caltrain_ridership[caltrain_ridership['Date Type'] != 'Holiday']


# Compute exposure: number of days of that Day_Type in the period
caltrain_ridership['exposure'] = caltrain_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Date Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
caltrain_ridership['Boardings'] = caltrain_ridership['Average Ridership'] * caltrain_ridership['exposure']

In [12]:
caltrain_ridership.sample(5)

,Month,Origin Station,Caltrain Ridership,Date Type,Average Ridership,start_date,end_date,exposure,Boardings
444,May 2024,San Martin,615.880266,Weekday,27.982745,2024-05-01,2024-05-31,23,643.603140
1304,June 2025,Millbrae,48188.156622,Sunday,1177.871295,2025-06-01,2025-06-30,5,5889.356477
848,December 2024,College Park,926.306378,Saturday,4.373056,2024-12-01,2024-12-31,4,17.492222
284,October 2024,Millbrae,37785.832128,Weekday,1353.336498,2024-10-01,2024-10-31,23,31126.739449
1209,December 2023,Gilroy,1429.774135,Saturday,0.000000,2023-12-01,2023-12-31,5,0.000000


In [13]:
culver_citybus_ridership.sample(5)

,Day Of Week,Route,Direction,Stop ID,Stop Name,AVG On,AVG Off,start_date,end_date,route_id,day_type
967,3-Sunday,1c1-Culver City Downtown Circulator,outbound,144,WashingtonBlvd/McmanusAve,0.0,0.0,2025-07-14,2025-08-25,1c1,Sunday
609,2-Saturday,1-Washington Boulevard,Inbound,136,Washington Blvd/Helms Ave,1.4,12.1,2025-07-14,2025-08-25,1,Saturday
849,2-Saturday,6-Sepulveda Boulevard,outbound,669,Sepulveda Blvd/Studio Village West,7.1,11.4,2025-07-14,2025-08-25,6,Saturday
369,1-Weekday,6-Sepulveda Boulevard,Inbound,624,Sepulveda Blvd/Braddock Dr,80.4,44.4,2025-07-14,2025-08-25,6,Weekday
1140,3-Sunday,6-Sepulveda Boulevard,outbound,665,SepulvedaBlvd/WashingtonBlvd,69.7,34.3,2025-07-14,2025-08-25,6,Sunday


In [14]:
group_cols = ['Day Of Week', 'Route', 'Stop ID', 'Stop Name', 
              'route_id', 'start_date', 'end_date', 'day_type']

# Sum AVG On and AVG Off
total_ridership_culver = culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg({
    'AVG On': 'sum'
})

Culver city : Missing records for either inbound or outbound for certain stops. For now, we are considering no service/no stop for that direction.

In [15]:
total_ridership_culver.head(5)

,Day Of Week,Route,Stop ID,Stop Name,route_id,start_date,end_date,day_type,AVG On
0,1-Weekday,1-Washington Boulevard,101,WindwardAve/MainSt,1,2025-07-14,2025-08-25,Weekday,111.2
1,1-Weekday,1-Washington Boulevard,102,Pacific Ave/N Venice Blvd,1,2025-07-14,2025-08-25,Weekday,31.7
2,1-Weekday,1-Washington Boulevard,103,Washington Blvd/Pacific Ave,1,2025-07-14,2025-08-25,Weekday,84.2
3,1-Weekday,1-Washington Boulevard,104,Washington Blvd/Via Dolce,1,2025-07-14,2025-08-25,Weekday,39.4
4,1-Weekday,1-Washington Boulevard,105,Washington Blvd/Via Marina,1,2025-07-14,2025-08-25,Weekday,42.4


In [16]:
# Compute exposure: number of days of that Day_Type in the period
total_ridership_culver['exposure'] = total_ridership_culver.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['day_type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_ridership_culver['Boardings'] = total_ridership_culver['AVG On'] * total_ridership_culver['exposure']

In [17]:
foothill_transit_ridership['day_type'].unique()

array(['weekday', 'holiday', 'weekend'], dtype=object)

In [18]:
foothill_transit_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694858 entries, 0 to 694857
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   date              694858 non-null  datetime64[ns]
 1   route_short_name  694858 non-null  int64         
 2   direction         694858 non-null  object        
 3   stop_code         694858 non-null  int64         
 4   stop_lat          694858 non-null  float64       
 5   stop_lon          694858 non-null  float64       
 6   boardings         694858 non-null  int64         
 7   alightings        694858 non-null  int64         
 8   start_date        694858 non-null  datetime64[ns]
 9   end_date          694858 non-null  datetime64[ns]
 10  day_type          694858 non-null  object        
dtypes: datetime64[ns](3), float64(2), int64(4), object(2)
memory usage: 58.3+ MB


In [19]:
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col='date')

In [20]:
group_cols = ['date', 'route_short_name', 'stop_code', 'stop_lat', 'stop_lon', 
              'start_date', 'end_date', 'day_type', 'Day Type']

# Sum AVG On and AVG Off
total_ridership_foothill = foothill_transit_ridership.groupby(
    group_cols, as_index=False).agg({
    'boardings': 'sum'
})

In [21]:
# Compute exposure: number of days of that Day_Type in the period
total_ridership_foothill['exposure'] = total_ridership_foothill.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_ridership_foothill['Boardings'] = total_ridership_foothill['boardings'] * total_ridership_foothill['exposure']

In [22]:
fresno_area_express_ridership.head(5)

,Date,StopID,StopLabel,ProjectedBoarding,ProjectedAlighting,start_date,end_date,day_type
0,2024-09-01,5,NE BRAWLEY - SHIELDS,44.691729,29.748092,2024-09-01,2024-09-01,weekend
1,2024-09-01,6,SE SHAW - BRAWLEY,7.000000,0.000000,2024-09-01,2024-09-01,weekend
2,2024-09-01,7,SW SHAW - WEST,20.000000,20.000000,2024-09-01,2024-09-01,weekend
3,2024-09-01,8,SE SHAW - BLACKSTONE,79.000000,17.000000,2024-09-01,2024-09-01,weekend
4,2024-09-01,9,SE SHAW - FIRST,63.000000,29.000000,2024-09-01,2024-09-01,weekend


bart_ridership, caltrain, fresno_area_express_ridership no route information

In [23]:
fresno_area_express_ridership['day_type'].unique()

array(['weekend', 'holiday', 'weekday'], dtype=object)

In [24]:
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col='Date')

# Compute exposure: number of days of that Day_Type in the period
fresno_area_express_ridership['exposure'] = fresno_area_express_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
fresno_area_express_ridership['Boardings'] = fresno_area_express_ridership['ProjectedBoarding'] * fresno_area_express_ridership['exposure']

In [25]:
fresno_area_express_ridership.head(5)

,Date,StopID,StopLabel,ProjectedBoarding,ProjectedAlighting,start_date,end_date,day_type,Day Type,exposure,Boardings
0,2024-09-01,5,NE BRAWLEY - SHIELDS,44.691729,29.748092,2024-09-01,2024-09-01,weekend,Sunday,1,44.691729
1,2024-09-01,6,SE SHAW - BRAWLEY,7.000000,0.000000,2024-09-01,2024-09-01,weekend,Sunday,1,7.000000
2,2024-09-01,7,SW SHAW - WEST,20.000000,20.000000,2024-09-01,2024-09-01,weekend,Sunday,1,20.000000
3,2024-09-01,8,SE SHAW - BLACKSTONE,79.000000,17.000000,2024-09-01,2024-09-01,weekend,Sunday,1,79.000000
4,2024-09-01,9,SE SHAW - FIRST,63.000000,29.000000,2024-09-01,2024-09-01,weekend,Sunday,1,63.000000


In [26]:
gold_coast_transit_ridership.head(5)

,day_of_week,route,direction,stop_id,unknown,stop_name,total_on,total_off,total_activity,cumulative_load,lat,lon,start_date,end_date
0,Weekday,1A,North,4THBST2,19,4th & B St,2,61,62,97,34.198975,-119.179621,2025-05-01,2025-05-31
1,Weekday,1A,South,4THBST1,1,4th & B St,70,2,72,188,34.199066,-119.179574,2025-05-01,2025-05-31
2,Weekday,1B,North,4THBST2,21,4th & B St,1,56,57,115,34.198975,-119.179621,2025-05-01,2025-05-31
3,Weekday,1B,South,4THBST1,1,4th & B St,63,2,65,183,34.199066,-119.179574,2025-05-01,2025-05-31
4,Weekday,1B,South,BAR5TH1,16,Bard & 5th,2,6,7,132,34.161169,-119.194368,2025-05-01,2025-05-31


In [27]:
gold_coast_transit_ridership['direction'].unique()

array(['North', 'South', 'Loop', 'East', 'West', 'PM', 'AM', 'A'],
      dtype=object)

Skipping this for now.
- Stop id is custom generated and cannot be used.
- Some rows don't have route numbers
- Will add this agency later

In [28]:
gold_coast_transit_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1046 entries, 0 to 1045
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   day_of_week      1046 non-null   object        
 1   route            1032 non-null   object        
 2   direction        1046 non-null   object        
 3   stop_id          1046 non-null   object        
 4   unknown          1046 non-null   int64         
 5   stop_name        1046 non-null   object        
 6   total_on         1046 non-null   int64         
 7   total_off        1046 non-null   int64         
 8   total_activity   1046 non-null   int64         
 9   cumulative_load  1046 non-null   int64         
 10  lat              1046 non-null   float64       
 11  lon              1046 non-null   float64       
 12  start_date       1046 non-null   datetime64[ns]
 13  end_date         1046 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(

In [29]:
golden_gate_park_shuttle_ridership['Day Type'].unique()

array(['Weekday', 'Weekend'], dtype=object)

In [30]:
golden_gate_park_shuttle_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6570 entries, 0 to 6569
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        6570 non-null   datetime64[ns]
 1   Month       6570 non-null   int64         
 2   Day Type    6570 non-null   object        
 3   Day         6570 non-null   object        
 4   Stop        6570 non-null   object        
 5   Ridership   6570 non-null   int64         
 6   start_date  6570 non-null   datetime64[ns]
 7   end_date    6570 non-null   datetime64[ns]
 8   direction   4380 non-null   object        
dtypes: datetime64[ns](3), int64(2), object(4)
memory usage: 462.1+ KB


In [31]:
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col='Date')

In [32]:
golden_gate_park_shuttle_ridership["Stop"] = golden_gate_park_shuttle_ridership["Stop"].str.replace(
    r"\s*-?\s*(EB|WB|NB|SB)$", 
    "", 
    regex=True
)

group_cols = [
    "Date", "Month", "Day Type", "Day", "Stop", "start_date", "end_date"]

# Sum AVG On and AVG Off
total_golden_gate_park_shuttle_ridership = golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg({
    'Ridership': 'sum'
})

In [33]:
# Compute exposure: number of days of that Day_Type in the period
total_golden_gate_park_shuttle_ridership['exposure'] = total_golden_gate_park_shuttle_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_golden_gate_park_shuttle_ridership['Boardings'] = total_golden_gate_park_shuttle_ridership['Ridership'] * total_golden_gate_park_shuttle_ridership['exposure']

In [34]:
total_golden_gate_park_shuttle_ridership.head(5)

,Date,Month,Day Type,Day,Stop,start_date,end_date,Ridership,exposure,Boardings
0,2024-07-01,7,Weekday,Monday,10th Ave/ De Young,2024-07-01,2024-07-01,9,1,9
1,2024-07-01,7,Weekday,Monday,8th Ave,2024-07-01,2024-07-01,7,1,7
2,2024-07-01,7,Weekday,Monday,Academy of Sciences,2024-07-01,2024-07-01,22,1,22
3,2024-07-01,7,Weekday,Monday,Blue Heron Boathouse,2024-07-01,2024-07-01,45,1,45
4,2024-07-01,7,Weekday,Monday,Conservatory of Flowers,2024-07-01,2024-07-01,20,1,20


In [35]:
golden_gate_transit_ridership.head(3)

,OPERATION_DATE,ROUTE,DIRECTION,STOP_NUMBER,STOP_NAME,BOARDINGS,ALIGHTINGS,date,start_date,end_date,day_type
0,01-SEP-25,101,North,40003,Salesforce Transit Center-Bus Plaza Bay A (40003),36,0,2025-09-01,2025-09-01,2025-09-01,holiday
1,01-SEP-25,101,North,40024,McAllister St & Polk St (40024),53,8,2025-09-01,2025-09-01,2025-09-01,holiday
2,01-SEP-25,101,North,40026,Van Ness Ave & Geary Blvd (40026),28,3,2025-09-01,2025-09-01,2025-09-01,holiday


In [36]:
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, 
                                             date_col='OPERATION_DATE')

group_cols = [
    "OPERATION_DATE", "ROUTE", "STOP_NAME", "STOP_NUMBER", 
    "date", "Day Type", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    Ridership = ('BOARDINGS', 'sum'))
)

/tmp/ipykernel_1110/1130206225.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], errors='coerce')


In [37]:
# Compute exposure: number of days of that Day_Type in the period
total_golden_gate_transit_ridership['exposure'] = total_golden_gate_transit_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['Day Type']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_golden_gate_transit_ridership['BOARDINGS'] = total_golden_gate_transit_ridership['Ridership'] * total_golden_gate_transit_ridership['exposure']

In [38]:
total_golden_gate_transit_ridership.head(5)

,OPERATION_DATE,ROUTE,STOP_NAME,STOP_NUMBER,date,Day Type,end_date,start_date,Ridership,exposure,BOARDINGS
0,2025-09-01,101,Commerce Blvd & Alison Ave (41231),41231,2025-09-01,Weekday,2025-09-01,2025-09-01,1,1,1
1,2025-09-01,101,Commerce Blvd & Arlen Dr (40884),40884,2025-09-01,Weekday,2025-09-01,2025-09-01,4,1,4
2,2025-09-01,101,Commerce Blvd & Arlen Dr (42042),42042,2025-09-01,Weekday,2025-09-01,2025-09-01,1,1,1
3,2025-09-01,101,Commerce Blvd & Enterprise Dr (40891),40891,2025-09-01,Weekday,2025-09-01,2025-09-01,0,1,0
4,2025-09-01,101,Commerce Blvd & Enterprise Dr (40892),40892,2025-09-01,Weekday,2025-09-01,2025-09-01,0,1,0


In [47]:
group_cols = [
    "DayType", "Route", "StopID", "StopName", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    Ridership = ('Boardings', 'sum'))
)

In [55]:
# Compute exposure: number of days of that Day_Type in the period
total_long_beach_transit_ridership['exposure'] = total_long_beach_transit_ridership.apply(
    lambda row: count_day_type_days(row['start_date'], row['end_date'], row['DayType']),
    axis=1
)

# Compute total boardings for the period (count for PPML)
total_long_beach_transit_ridership['BOARDINGS'] = total_long_beach_transit_ridership['Ridership'] * total_long_beach_transit_ridership['exposure']


In [56]:
total_long_beach_transit_ridership.head(5)

,DayType,Route,StopID,StopName,end_date,start_date,Ridership,exposure,BOARDINGS
0,Saturday,1,2001,2727 Del Amo Blvd N,2025-06-30,2024-07-01,2.582828,52,134.307032
1,Saturday,1,2002,2660 Del Amo Blvd S,2025-06-30,2024-07-01,0.000000,52,0.000000
2,Saturday,1,2003,Del Amo & Rancho Way NW,2025-06-30,2024-07-01,2.282203,52,118.674575
3,Saturday,1,2004,Del Amo & Fordyce SW,2025-06-30,2024-07-01,5.977199,52,310.814362
4,Saturday,1,2005,Del Amo & Wilmington NW,2025-06-30,2024-07-01,3.527042,52,183.406161


In [58]:
omni_trans_ridership.head(5)

,FiscalYear,Route,Stop Name,Total Board,Total Alight,avg_boardings,avg_alightings,start_date,end_date,day_type
0,2024,1,2ND @ F ST,939,226,2.572603,0.619178,2023-07-01,2024-06-30,all
1,2024,1,2ND @ G ST,515,2811,1.410959,7.701370,2023-07-01,2024-06-30,all
2,2024,1,2ND @ J ST,1005,458,2.753425,1.254795,2023-07-01,2024-06-30,all
3,2024,1,2ND @ LST,3958,3366,10.843836,9.221918,2023-07-01,2024-06-30,all
4,2024,1,2nd @ 215,600,948,1.643836,2.597260,2023-07-01,2024-06-30,all


Omni trans doesnt have weekday, saturday and sunday distinction so, we are not using this at the moment.

In [66]:
riverside_transit_ridership.head(5)

,date,Stop ID,Route,Direction,ridership,Longitude,Latitude,start_date,end_date,day_type,Day Type
0,2025-05-01,0,1,Inbound,69,-117.594376,33.982152,2025-05-01,2025-05-01,weekday,Weekday
1,2025-05-01,0,3,Inbound,9,-117.564320,33.881796,2025-05-01,2025-05-01,weekday,Weekday
2,2025-05-01,0,3,Outbound,2,-117.554640,33.994648,2025-05-01,2025-05-01,weekday,Weekday
3,2025-05-01,0,8,Inbound,3,-117.334352,33.695128,2025-05-01,2025-05-01,weekday,Weekday
4,2025-05-01,0,16,Inbound,27,-117.339768,33.979044,2025-05-01,2025-05-01,weekday,Weekday


In [65]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    date	Stop ID	Route	Direction	ridership	Longitude	Latitude	start_date	end_date	day_type	Day Type]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    Ridership = ('Boardings', 'sum'))
)
